# Bone Fracture Detection — Master Notebook

IEEE Access 2025 stack + YOLOv8. Runs on **local Jupyter** or **Google Colab**.

| Stage | Model | Benchmark |
| --- | --- | --- |
| 1 Localization | YOLOv8 | mAP50 = 0.86 |
| 2a | VGG-16 Softmax | acc 0.95 |
| 2b | VGG-16 + Random Forest | acc 0.95 |
| 2c | ResNet-50 + SVM | acc 0.93 |
| 2d | EfficientNetB0 + XGBoost | acc ~0.41 |


In [ ]:
import sys, pathlib
IN_COLAB = "google.colab" in sys.modules
ROOT = pathlib.Path("/content") if IN_COLAB else pathlib.Path("..").resolve()
if not (ROOT / "main.py").exists():
    ROOT = pathlib.Path(".").resolve()
sys.path.insert(0, str(ROOT))
print("ROOT", ROOT, "Colab", IN_COLAB)


In [ ]:
%pip install -q flask opencv-python-headless scikit-learn xgboost matplotlib pandas pydicom
# On Colab also: %pip install -q tensorflow ultralytics


In [ ]:
from config import Config
from dataset_handler.preprocess import make_sample_structure, dataset_scale_report
from explainability.metrics import evaluate_classifier, plot_roc
import numpy as np

Config.ensure_directories()
root = make_sample_structure(Config.DATASET_DIR / "classification", per_class=2)
print(dataset_scale_report(root / "train", root / "val"))


## Cross-check public 20k-scale sources (no 15 GB download)

Probes Kaggle + Figshare and verifies GRAZPEDWRI-DX CSV has 20,327 rows.


In [ ]:
from dataset_handler.catalog import probe_and_report
try:
    probe_and_report(probe=True)
except Exception as exc:
    print("Probe skipped:", exc)


## Metric engine (Precision, Recall, F1, Specificity, Accuracy, GFLOPs, ROC)

Uses a synthetic hold-out so this cell runs without trained weights. Replace `y_true`/`y_proba` with validation predictions after `python main.py train`.


In [ ]:
rng = np.random.default_rng(1)
C = len(Config.MORPHOLOGY_CLASSES)
y_true = rng.integers(0, C, 300)
y_pred = y_true.copy()
mask = rng.random(300) < 0.07
y_pred[mask] = (y_pred[mask] + 1) % C
proba = np.eye(C)[y_pred] * 0.75 + rng.random((300, C)) * 0.25
proba /= proba.sum(1, keepdims=True)
rep = evaluate_classifier(y_true, y_pred, proba, Config.MORPHOLOGY_CLASSES, "VGG-16")
print({k: round(rep[k], 4) if isinstance(rep[k], float) else rep[k] for k in
       ["accuracy","precision_macro","recall_macro","f1_macro","specificity_macro","roc_auc_ovr_macro","gflops_256"]})
plot_roc(y_true, proba, Config.MORPHOLOGY_CLASSES, Config.METRICS_DIR / "roc_master.png")
print("ROC ->", Config.METRICS_DIR / "roc_master.png")


## Two-stage inference + Grad-CAM + pre/post refixation on a synthetic X-ray


In [ ]:
from models.pipeline import PIPELINE
sample = next((Config.DATASET_DIR / "classification" / "train" / "Comminuted Fracture").glob("*.png"))
result = PIPELINE.analyze(sample, "notebook_demo")
print(result.predicted_class, result.anatomical_site, result.needs_refixation, result.refix_rel)


## Train (uncomment on a GPU runtime after real ImageFolder data is present)

```python
# !python main.py train --model softmax --epochs 20
# !python main.py train --model rf
```
